In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP", "default")
RPT_SMALL_URL = os.getenv("RPT_SMALL_URL")
RPT_LARGE_URL = os.getenv("RPT_LARGE_URL")

os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

import requests
import json

def get_ai_core_token():
    """
    Get OAuth2 token from AI Core authentication endpoint.
    
    Returns:
        str: Bearer token for AI Core API calls
    """
    auth_url = os.getenv("AICORE_AUTH_URL")
    client_id = os.getenv("AICORE_CLIENT_ID")
    client_secret = os.getenv("AICORE_CLIENT_SECRET")
    
    if not all([auth_url, client_id, client_secret]):
        raise ValueError("AI Core credentials not found in environment variables")
    
    # Request token
    token_response = requests.post(
        f"{auth_url}/oauth/token",
        auth=(client_id, client_secret),
        data={"grant_type": "client_credentials"}
    )
    token_response.raise_for_status()
    
    return token_response.json()["access_token"]

def predict_with_rpt1(rows, index_column=None, target_columns=None, data_schema=None, model_size="small", deployment_url=None):
    """
    Make predictions using SAP-RPT-1 Model deployed on AI Core.
    
    Args:
        rows (list): List of dictionaries representing data rows
                    Context rows should have complete data
                    Query rows should have '[PREDICT]' for values to predict
        index_column (str, optional): Column name to use as row identifier
        target_columns (list, optional): List of dicts with target column configuration
                    Example: [{"name": "COSTCENTER", "prediction_placeholder": "[PREDICT]", "task_type": "classification"}]
                    If not provided, will auto-detect from rows
        data_schema (dict, optional): Schema defining column data types
                    Example: {"PRODUCT": {"dtype": "string"}, "PRICE": {"dtype": "numeric"}}
        model_size (str, optional): Model size to use - "small" or "large". Default is "small"
        deployment_url (str, optional): AI Core deployment URL. If not provided, uses RPT_SMALL_URL or RPT_LARGE_URL based on model_size
    
    Returns:
        dict: Prediction results from the API
    
    Example:
        rows = [
            {"ID": "35", "PRODUCT": "Couch", "PRICE": 999.99, "COSTCENTER": "[PREDICT]"},
            {"ID": "44", "PRODUCT": "Office Chair", "PRICE": 150.8, "COSTCENTER": "Office Furniture"}
        ]
        target_columns = [{"name": "COSTCENTER", "prediction_placeholder": "[PREDICT]", "task_type": "classification"}]
        result = predict_with_rpt1(rows, index_column="ID", target_columns=target_columns, model_size="large")
    """
    
    # Get deployment URL based on model size
    if not deployment_url:
        if model_size.lower() == "small":
            deployment_url = os.getenv("RPT_SMALL_URL")
        elif model_size.lower() == "large":
            deployment_url = os.getenv("RPT_LARGE_URL")
        else:
            raise ValueError(f"Invalid model_size '{model_size}'. Must be 'small' or 'large'")
    
    if not deployment_url:
        raise ValueError(f"Deployment URL not found. Please set RPT_{model_size.upper()}_URL in environment variables")
    
    # Ensure URL ends with /predict
    if not deployment_url.endswith('/predict'):
        deployment_url = deployment_url.rstrip('/') + '/predict'
    
    # Get AI Core token
    try:
        auth_token = get_ai_core_token()
    except Exception as e:
        print(f"Failed to get AI Core token: {e}")
        return None
    
    # Auto-detect target columns if not provided
    if not target_columns:
        target_columns = []
        if rows:
            for key, value in rows[0].items():
                if value == "[PREDICT]":
                    target_columns.append({
                        "name": key,
                        "prediction_placeholder": "[PREDICT]",
                        "task_type": "classification"  # default, could be inferred better
                    })
    
    # Prepare request payload
    payload = {
        "rows": rows,
        "prediction_config": {
            "target_columns": target_columns
        }
    }
    
    if index_column:
        payload["index_column"] = index_column
    
    if data_schema:
        payload["data_schema"] = data_schema
    
    # Prepare headers
    headers = {
        "Authorization": f"Bearer {auth_token}",
        "AI-Resource-Group": os.getenv("AICORE_RESOURCE_GROUP", "default"),
        "Content-Type": "application/json"
    }
    
    try:
        # Make API request
        print(f"Using RPT-1 {model_size.upper()} model...")
        response = requests.post(deployment_url, headers=headers, json=payload)
        
        # Check for rate limiting
        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After', 'unknown')
            print(f"Rate limit exceeded. Retry after {retry_after} seconds.")
            return None
        
        # Check for service unavailability
        if response.status_code == 503:
            retry_after = response.headers.get('Retry-After', 'unknown')
            print(f"Service unavailable. Retry after {retry_after} seconds.")
            return None
        
        # Raise error for bad status codes
        response.raise_for_status()
        
        # Return parsed response
        return response.json()
        
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        if hasattr(e, 'response') and hasattr(e.response, 'text'):
            print(f"Response: {e.response.text}")
        return None

In [7]:
rows = [
    {"ID": "35", "PRODUCT": "Couch", "PRICE": "[PREDICT]", "COSTCENTER": "Living Room"},
    {"ID": "44", "PRODUCT": "Office Chair", "PRICE": 150.8, "COSTCENTER": "Office Furniture"},
    {"ID": "27", "PRODUCT": "Sofa", "PRICE": 320.3, "COSTCENTER": "Living Room"},
    {"ID": "38", "PRODUCT": "Table", "PRICE": 129.6, "COSTCENTER": "Office Furniture"},
    {"ID": "56", "PRODUCT": "Lamp", "PRICE": 23.7, "COSTCENTER": "Office Furniture"}
]

target_columns = [{
    "name": "PRICE",
    "prediction_placeholder": "[PREDICT]",
    "task_type": "regression"
}]

result = predict_with_rpt1(rows, index_column="ID", target_columns=target_columns)
result

Using RPT-1 SMALL model...


{'id': '23bfa88a-19a6-4247-858b-305c97eea1b8',
 'metadata': {'num_columns': 4,
  'num_predictions': 1,
  'num_query_rows': 1,
  'num_rows': 4},
 'predictions': [{'ID': 35,
   'PRICE': [{'confidence': None, 'prediction': 311.9784240722656}]}],
 'status': {'code': 0, 'message': 'ok'}}